In [6]:
import os
import warnings
import pandas as pd
import numpy as np
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from joblib import Parallel, delayed
from funcs.engineer_features_funcs import (
    compute_mse_scores, 
    evaluate_feature, 
    compute_mse_with_added_feature, 
    compute_mse_with_dropped_feature
)

# Suppress warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Adjust Optuna logging level
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Paths for processed data and TSFRESH features
processed_train_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\processed\train_transformed_combined.csv'
processed_test_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\processed\test_transformed_combined.csv'
tsfresh_train_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\train_combined_all_features_filled.csv'
tsfresh_test_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\test_combined_all_features_filled.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)
tsfresh_features_train = pd.read_csv(tsfresh_train_path, index_col='Date', parse_dates=True)
tsfresh_features_test = pd.read_csv(tsfresh_test_path, index_col='Date', parse_dates=True)

# Define the base features
base_features = []

# Targets to evaluate
targets = ['FEDFUNDS']

# Model parameters
xgboost_params = {'max_depth': 9, 'learning_rate': 0.0168, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5866, 'colsample_bytree': 0.9177, 'reg_alpha': 0.0125, 'reg_lambda': 8.577e-05, 'verbosity': 0}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402, 'colsample_bytree': 0.9405, 'reg_alpha': 0.000756, 'reg_lambda': 0.000259, 'verbosity': -1}

# Threshold for feature dropping
relative_threshold = 0.002

# Loop through each target
results = []

for target in targets:
    print(f"Processing target: {target}")
    
    X_train = pd.DataFrame(train_combined[target])
    X_test = pd.DataFrame(test_combined[target])
    y_train = train_combined[[target]]
    y_test = test_combined[[target]]
    
    all_added_features = []

    baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [target])
    print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")

    for round_num in range(5):
        print(f"\nRound {round_num + 1} of feature addition")

        results_added = Parallel(n_jobs=-1)(delayed(evaluate_feature)(
            feature, tsfresh_features_train, tsfresh_features_test, X_train, X_test,
            y_train, y_test, all_added_features, aggregated_baseline_mse, all_added_features
        ) for feature in tsfresh_features_train.columns)
        
        results_added = [res for res in results_added if res is not None]
        results_added.sort(key=lambda x: x[1])

        top_to_add = [f for f in results_added[:3] if f[2] > 0]
        for feature, _, improvement, _, _ in top_to_add:
            all_added_features.append(feature)
            X_train[feature] = tsfresh_features_train[feature]
            X_test[feature] = tsfresh_features_test[feature]
            print(f"Added feature: {feature} with improvement: {improvement}")

        new_mse_scores, aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [target] + all_added_features)
        print(f"New aggregated MSE after addition: {aggregated_mse}")

        print(f"\nRound {round_num + 1} of feature dropping")
        results_dropped = Parallel(n_jobs=-1)(delayed(compute_mse_with_dropped_feature)(
            X_train, X_test, y_train, y_test, [target] + all_added_features, feature
        ) for feature in all_added_features)

        results_dropped.sort(key=lambda x: x[1])
        top_to_drop = [f for f in results_dropped if f[2] > threshold]

        for feature, _, improvement, _, _ in top_to_drop:
            all_added_features.remove(feature)
            X_train.drop(columns=[feature], inplace=True)
            X_test.drop(columns=[feature], inplace=True)
            print(f"Dropped feature: {feature} with improvement: {improvement}")

        final_mse_scores, final_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [target] + all_added_features)
        print(f"Final aggregated MSE after dropping: {final_aggregated_mse}")

    results.append({
        'target': target,
        'initial_mse_xgboost': baseline_mse_scores['XGBoost'],
        'initial_mse_lightgbm': baseline_mse_scores['LightGBM'],
        'final_mse_xgboost': final_mse_scores['XGBoost'],
        'final_mse_lightgbm': final_mse_scores['LightGBM'],
        'initial_aggregated_mse': aggregated_baseline_mse,
        'final_aggregated_mse': final_aggregated_mse,
        'improvement': aggregated_baseline_mse - final_aggregated_mse,
        'final_feature_space': [target] + all_added_features
    })

    features_df = pd.DataFrame({'features': [target] + all_added_features})
    features_df.to_csv(f'data/engineered/{target}_final_features.csv', index=False)

results_df = pd.DataFrame(results)
results_df.to_csv('feature_engineering_results.csv', index=False)

print("Feature engineering completed for all targets.")


Processing target: FEDFUNDS
Initial aggregated baseline MSE: 0.00133645302582609

Round 1 of feature addition
New aggregated MSE after addition: 0.0012454051628824592

Round 1 of feature dropping
Final aggregated MSE after dropping: 0.001674194700801291

Round 2 of feature addition


KeyboardInterrupt: 

In [7]:
import os
import warnings
import pandas as pd
import numpy as np
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from joblib import Parallel, delayed
from funcs.engineer_features_funcs import (
    compute_mse_scores, 
    evaluate_feature, 
    compute_mse_with_added_feature, 
    compute_mse_with_dropped_feature
)

# Suppress warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Adjust Optuna logging level
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Paths for processed data and TSFRESH features
processed_train_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\processed\train_transformed_combined.csv'
processed_test_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\processed\test_transformed_combined.csv'
tsfresh_train_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\train_combined_all_features_filled.csv'
tsfresh_test_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\test_combined_all_features_filled.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)
tsfresh_features_train = pd.read_csv(tsfresh_train_path, index_col='Date', parse_dates=True)
tsfresh_features_test = pd.read_csv(tsfresh_test_path, index_col='Date', parse_dates=True)

# Define the base features
base_features = []

# Targets to evaluate
targets = ['FEDFUNDS']

# Model parameters
xgboost_params = {'max_depth': 9, 'learning_rate': 0.0168, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5866, 'colsample_bytree': 0.9177, 'reg_alpha': 0.0125, 'reg_lambda': 8.577e-05, 'verbosity': 0}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402, 'colsample_bytree': 0.9405, 'reg_alpha': 0.000756, 'reg_lambda': 0.000259, 'verbosity': -1}

# Relative threshold for feature dropping
relative_threshold = 0.002

# Loop through each target
results = []

for target in targets:
    print(f"Processing target: {target}")
    
    X_train = pd.DataFrame(train_combined[target])
    X_test = pd.DataFrame(test_combined[target])
    y_train = train_combined[[target]]
    y_test = test_combined[[target]]
    
    all_added_features = []

    baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [target])
    print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")

    for round_num in range(5):
        print(f"\nRound {round_num + 1} of feature addition")

        results_added = Parallel(n_jobs=-1)(delayed(evaluate_feature)(
            feature, tsfresh_features_train, tsfresh_features_test, X_train, X_test,
            y_train, y_test, all_added_features, aggregated_baseline_mse, all_added_features
        ) for feature in tsfresh_features_train.columns)
        
        results_added = [res for res in results_added if res is not None]
        results_added.sort(key=lambda x: x[1])

        top_to_add = [f for f in results_added[:3] if f[2] > 0]
        for feature, _, improvement, _, _ in top_to_add:
            all_added_features.append(feature)
            X_train[feature] = tsfresh_features_train[feature]
            X_test[feature] = tsfresh_features_test[feature]
            print(f"Added feature: {feature} with improvement: {improvement}")

        new_mse_scores, aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [target] + all_added_features)
        print(f"New aggregated MSE after addition: {aggregated_mse}")

        # Calculate the dynamic threshold for this round
        threshold = relative_threshold * aggregated_baseline_mse

        print(f"\nRound {round_num + 1} of feature dropping")
        results_dropped = Parallel(n_jobs=-1)(delayed(compute_mse_with_dropped_feature)(
            X_train, X_test, y_train, y_test, [target] + all_added_features, feature
        ) for feature in all_added_features)

        results_dropped.sort(key=lambda x: x[1])
        top_to_drop = [f for f in results_dropped if f[2] > threshold]

        for feature, _, improvement, _, _ in top_to_drop:
            all_added_features.remove(feature)
            X_train.drop(columns=[feature], inplace=True)
            X_test.drop(columns=[feature], inplace=True)
            print(f"Dropped feature: {feature} with improvement: {improvement}")

        final_mse_scores, final_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [target] + all_added_features)
        print(f"Final aggregated MSE after dropping: {final_aggregated_mse}")

    results.append({
        'target': target,
        'initial_mse_xgboost': baseline_mse_scores['XGBoost'],
        'initial_mse_lightgbm': baseline_mse_scores['LightGBM'],
        'final_mse_xgboost': final_mse_scores['XGBoost'],
        'final_mse_lightgbm': final_mse_scores['LightGBM'],
        'initial_aggregated_mse': aggregated_baseline_mse,
        'final_aggregated_mse': final_aggregated_mse,
        'improvement': aggregated_baseline_mse - final_aggregated_mse,
        'final_feature_space': [target] + all_added_features
    })

    features_df = pd.DataFrame({'features': [target] + all_added_features})
    features_df.to_csv(f'data/engineered/{target}/{target}_final_features.csv', index=False)

results_df = pd.DataFrame(results)
results_df.to_csv('feature_engineering_results.csv', index=False)

print("Feature engineering completed for all targets.")


Processing target: FEDFUNDS
Initial aggregated baseline MSE: 0.001016632373221907

Round 1 of feature addition
New aggregated MSE after addition: 0.0011807405103868628

Round 1 of feature dropping
Final aggregated MSE after dropping: 0.001329806228718473

Round 2 of feature addition


KeyboardInterrupt: 

In [8]:
import os
import warnings
import pandas as pd
import numpy as np
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from joblib import Parallel, delayed
from funcs.engineer_features_funcs import (
    compute_mse_scores, 
    evaluate_feature, 
    compute_mse_with_added_feature, 
    compute_mse_with_dropped_feature
)

# Suppress warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Adjust Optuna logging level
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Paths for processed data and TSFRESH features
processed_train_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\processed\train_transformed_combined.csv'
processed_test_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\processed\test_transformed_combined.csv'
tsfresh_train_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\train_combined_all_features_filled.csv'
tsfresh_test_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\test_combined_all_features_filled.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)
tsfresh_features_train = pd.read_csv(tsfresh_train_path, index_col='Date', parse_dates=True)
tsfresh_features_test = pd.read_csv(tsfresh_test_path, index_col='Date', parse_dates=True)

# Define the base features
base_features = []

# Targets to evaluate
targets = ['FEDFUNDS']

# Model parameters
xgboost_params = {'max_depth': 9, 'learning_rate': 0.0168, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5866, 'colsample_bytree': 0.9177, 'reg_alpha': 0.0125, 'reg_lambda': 8.577e-05, 'verbosity': 0}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402, 'colsample_bytree': 0.9405, 'reg_alpha': 0.000756, 'reg_lambda': 0.000259, 'verbosity': -1}

# Relative threshold for feature dropping
relative_threshold = 0.002

# Loop through each target
results = []

for target in targets:
    print(f"Processing target: {target}")
    
    X_train = pd.DataFrame(train_combined[target])
    X_test = pd.DataFrame(test_combined[target])
    y_train = train_combined[[target]]
    y_test = test_combined[[target]]
    
    all_added_features = []

    # Initial baseline MSE calculation
    baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [target])
    print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")

    for round_num in range(5):
        print(f"\nRound {round_num + 1} of feature addition")

        # Evaluate adding features
        results_added = Parallel(n_jobs=-1)(delayed(evaluate_feature)(
            feature, tsfresh_features_train, tsfresh_features_test, X_train, X_test,
            y_train, y_test, all_added_features, aggregated_baseline_mse, all_added_features
        ) for feature in tsfresh_features_train.columns)
        
        results_added = [res for res in results_added if res is not None]
        results_added.sort(key=lambda x: x[1])

        top_to_add = [f for f in results_added[:3] if f[2] > 0]
        for feature, _, improvement, _, _ in top_to_add:
            all_added_features.append(feature)
            X_train[feature] = tsfresh_features_train[feature]
            X_test[feature] = tsfresh_features_test[feature]
            print(f"Added feature: {feature} with improvement: {improvement}")

        # Recalculate the baseline MSE after feature addition
        baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [target] + all_added_features)
        print(f"New aggregated MSE after addition: {aggregated_baseline_mse}")

        # Calculate the dynamic threshold for this round
        threshold = relative_threshold * aggregated_baseline_mse

        print(f"\nRound {round_num + 1} of feature dropping")

        # Evaluate dropping features
        results_dropped = Parallel(n_jobs=-1)(delayed(compute_mse_with_dropped_feature)(
            X_train, X_test, y_train, y_test, [target] + all_added_features, feature
        ) for feature in all_added_features)

        results_dropped.sort(key=lambda x: x[1])
        top_to_drop = [f for f in results_dropped if f[2] > threshold]

        # Debugging print to understand the behavior
        print(f"Features considered for dropping (with their improvements): {[f[0] for f in top_to_drop]}")

        for feature, _, improvement, _, _ in top_to_drop:
            if improvement > threshold:
                all_added_features.remove(feature)
                X_train.drop(columns=[feature], inplace=True)
                X_test.drop(columns=[feature], inplace=True)
                print(f"Dropped feature: {feature} with improvement: {improvement}")

        # Recalculate the baseline MSE after feature dropping
        final_mse_scores, final_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [target] + all_added_features)
        print(f"Final aggregated MSE after dropping: {final_aggregated_mse}")

    results.append({
        'target': target,
        'initial_mse_xgboost': baseline_mse_scores['XGBoost'],
        'initial_mse_lightgbm': baseline_mse_scores['LightGBM'],
        'final_mse_xgboost': final_mse_scores['XGBoost'],
        'final_mse_lightgbm': final_mse_scores['LightGBM'],
        'initial_aggregated_mse': aggregated_baseline_mse,
        'final_aggregated_mse': final_aggregated_mse,
        'improvement': aggregated_baseline_mse - final_aggregated_mse,
        'final_feature_space': [target] + all_added_features
    })

    features_df = pd.DataFrame({'features': [target] + all_added_features})
    features_df.to_csv(f'data/engineered/{target}_final_features.csv', index=False)

results_df = pd.DataFrame(results)
results_df.to_csv('feature_engineering_results.csv', index=False)

print("Feature engineering completed for all targets.")


Processing target: FEDFUNDS
Initial aggregated baseline MSE: 0.0009850738382623128

Round 1 of feature addition
New aggregated MSE after addition: 0.0011759606020056977

Round 1 of feature dropping
Features considered for dropping (with their improvements): []
Final aggregated MSE after dropping: 0.001295193500175835

Round 2 of feature addition


KeyboardInterrupt: 

In [9]:
import os
import warnings
import pandas as pd
import numpy as np
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from joblib import Parallel, delayed
from funcs.engineer_features_funcs import (
    compute_mse_scores, 
    evaluate_feature, 
    compute_mse_with_added_feature, 
    compute_mse_with_dropped_feature
)

# Suppress warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Adjust Optuna logging level
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Paths for processed data and TSFRESH features
processed_train_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\processed\train_transformed_combined.csv'
processed_test_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\processed\test_transformed_combined.csv'
tsfresh_train_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\train_combined_all_features_filled.csv'
tsfresh_test_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\test_combined_all_features_filled.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)
tsfresh_features_train = pd.read_csv(tsfresh_train_path, index_col='Date', parse_dates=True)
tsfresh_features_test = pd.read_csv(tsfresh_test_path, index_col='Date', parse_dates=True)

# Define the base features
base_features = []

# Targets to evaluate
targets = ['FEDFUNDS']

# Model parameters
xgboost_params = {'max_depth': 9, 'learning_rate': 0.0168, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5866, 'colsample_bytree': 0.9177, 'reg_alpha': 0.0125, 'reg_lambda': 8.577e-05, 'verbosity': 0}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402, 'colsample_bytree': 0.9405, 'reg_alpha': 0.000756, 'reg_lambda': 0.000259, 'verbosity': -1}

# Relative threshold for feature dropping
relative_threshold = 0.002

# Loop through each target
results = []

for target in targets:
    print(f"\nProcessing target: {target}")
    
    X_train = pd.DataFrame(train_combined[target])
    X_test = pd.DataFrame(test_combined[target])
    y_train = train_combined[[target]]
    y_test = test_combined[[target]]
    
    all_added_features = []

    # Initial baseline MSE calculation
    baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [target])
    print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
    print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
    print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

    for round_num in range(5):
        print(f"\n---- Round {round_num + 1} of feature addition ----")

        # Evaluate adding features
        results_added = Parallel(n_jobs=-1)(delayed(evaluate_feature)(
            feature, tsfresh_features_train, tsfresh_features_test, X_train, X_test,
            y_train, y_test, all_added_features, aggregated_baseline_mse, all_added_features
        ) for feature in tsfresh_features_train.columns)
        
        results_added = [res for res in results_added if res is not None]
        results_added.sort(key=lambda x: x[1])

        if results_added:
            print(f"Top features considered for addition: {[f[0] for f in results_added[:3]]}")
        
        top_to_add = [f for f in results_added[:3] if f[2] > 0]
        for feature, _, improvement, _, _ in top_to_add:
            all_added_features.append(feature)
            X_train[feature] = tsfresh_features_train[feature]
            X_test[feature] = tsfresh_features_test[feature]
            print(f"Added feature: {feature} with improvement: {improvement}")

        # Recalculate the baseline MSE after feature addition
        baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [target] + all_added_features)
        print(f"New aggregated MSE after addition: {aggregated_baseline_mse}")
        print(f"New MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
        print(f"New MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

        # Calculate the dynamic threshold for this round
        threshold = relative_threshold * aggregated_baseline_mse

        print(f"\n---- Round {round_num + 1} of feature dropping ----")

        # Evaluate dropping features
        results_dropped = Parallel(n_jobs=-1)(delayed(compute_mse_with_dropped_feature)(
            X_train, X_test, y_train, y_test, [target] + all_added_features, feature
        ) for feature in all_added_features)

        results_dropped.sort(key=lambda x: x[1])

        if results_dropped:
            print(f"Features considered for dropping (with their improvements): {[f[0] for f in results_dropped]}")

        top_to_drop = [f for f in results_dropped if f[2] > threshold]

        for feature, _, improvement, _, _ in top_to_drop:
            if improvement > threshold:
                all_added_features.remove(feature)
                X_train.drop(columns=[feature], inplace=True)
                X_test.drop(columns=[feature], inplace=True)
                print(f"Dropped feature: {feature} with improvement: {improvement}")

        # Recalculate the baseline MSE after feature dropping
        final_mse_scores, final_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [target] + all_added_features)
        print(f"Final aggregated MSE after dropping: {final_aggregated_mse}")
        print(f"Final MSE for XGBoost: {final_mse_scores['XGBoost']}")
        print(f"Final MSE for LightGBM: {final_mse_scores['LightGBM']}")

    results.append({
        'target': target,
        'initial_mse_xgboost': baseline_mse_scores['XGBoost'],
        'initial_mse_lightgbm': baseline_mse_scores['LightGBM'],
        'final_mse_xgboost': final_mse_scores['XGBoost'],
        'final_mse_lightgbm': final_mse_scores['LightGBM'],
        'initial_aggregated_mse': aggregated_baseline_mse,
        'final_aggregated_mse': final_aggregated_mse,
        'improvement': aggregated_baseline_mse - final_aggregated_mse,
        'final_feature_space': [target] + all_added_features
    })

    features_df = pd.DataFrame({'features': [target] + all_added_features})
    features_df.to_csv(f'data/engineered/{target}_final_features.csv', index=False)

results_df = pd.DataFrame(results)
results_df.to_csv('feature_engineering_results.csv', index=False)

print("Feature engineering completed for all targets.")



Processing target: FEDFUNDS
Initial aggregated baseline MSE: 0.0014163311950126753
Initial MSE for XGBoost: 0.00045064907221405604
Initial MSE for LightGBM: 0.0009656821227986193

---- Round 1 of feature addition ----


KeyboardInterrupt: 

In [4]:
import os
import warnings
import pandas as pd
import numpy as np
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from joblib import Parallel, delayed
from funcs.engineer_features_funcs import (
    compute_mse_scores, 
    evaluate_feature, 
    compute_mse_with_added_feature, 
    compute_mse_with_dropped_feature
)

# Suppress warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Adjust Optuna logging level
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Paths for processed data and TSFRESH features
processed_train_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\processed\train_transformed_combined.csv'
processed_test_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\processed\test_transformed_combined.csv'
tsfresh_train_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\train_combined_all_features_filled.csv'
tsfresh_test_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\test_combined_all_features_filled.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)
tsfresh_features_train = pd.read_csv(tsfresh_train_path, index_col='Date', parse_dates=True)
tsfresh_features_test = pd.read_csv(tsfresh_test_path, index_col='Date', parse_dates=True)

# Define the base features
base_features = []

# Targets to evaluate
targets = ['FEDFUNDS']

# Model parameters
xgboost_params = {'max_depth': 9, 'learning_rate': 0.0168, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5866, 'colsample_bytree': 0.9177, 'reg_alpha': 0.0125, 'reg_lambda': 8.577e-05, 'verbosity': 0}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402, 'colsample_bytree': 0.9405, 'reg_alpha': 0.000756, 'reg_lambda': 0.000259, 'verbosity': -1}

# Relative threshold for feature dropping
relative_threshold = 0.002

# Loop through each target
results = []

for target in targets:
    print(f"\nProcessing target: {target}")
    columns = list(train_combined.columns)
    X_train = train_combined.drop(columns=columns)
    X_test = test_combined.drop(columns=['FEDFUNDS'])
    y_train = train_combined[[target]]
    y_test = test_combined[[target]]
    display(X_train)
    display(y_train)

    
    all_added_features = []

    # Initial baseline MSE calculation
    baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [target])
    print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
    print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
    print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

    for round_num in range(5):
        print(f"\n---- Round {round_num + 1} of feature addition ----")

        # Evaluate adding features
        results_added = Parallel(n_jobs=-1)(delayed(evaluate_feature)(
            feature, tsfresh_features_train, tsfresh_features_test, X_train, X_test,
            y_train, y_test, all_added_features, aggregated_baseline_mse, all_added_features
        ) for feature in tsfresh_features_train.columns)
        
        results_added = [res for res in results_added if res is not None]
        results_added.sort(key=lambda x: x[1])

        if results_added:
            print(f"Top features considered for addition: {[f[0] for f in results_added[:3]]}")
        
        top_to_add = [f for f in results_added[:3] if f[2] > 0]
        for feature, _, improvement, _, _ in top_to_add:
            all_added_features.append(feature)
            X_train[feature] = tsfresh_features_train[feature]
            X_test[feature] = tsfresh_features_test[feature]
            print(f"Added feature: {feature} with improvement: {improvement}")

        # Recalculate the baseline MSE after feature addition
        baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [target] + all_added_features)
        print(f"New aggregated MSE after addition: {aggregated_baseline_mse}")
        print(f"New MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
        print(f"New MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

        # Calculate the dynamic threshold for this round
        threshold = relative_threshold * aggregated_baseline_mse

        print(f"\n---- Round {round_num + 1} of feature dropping ----")

        # Evaluate dropping features
        results_dropped = Parallel(n_jobs=-1)(delayed(compute_mse_with_dropped_feature)(
            X_train, X_test, y_train, y_test, [target] + all_added_features, feature
        ) for feature in all_added_features)

        results_dropped.sort(key=lambda x: x[1])

        if results_dropped:
            print(f"Features considered for dropping (with their improvements): {[f[0] for f in results_dropped]}")

        top_to_drop = [f for f in results_dropped if f[2] > threshold]

        for feature, _, improvement, _, _ in top_to_drop:
            if improvement > threshold:
                all_added_features.remove(feature)
                X_train.drop(columns=[feature], inplace=True)
                X_test.drop(columns=[feature], inplace=True)
                print(f"Dropped feature: {feature} with improvement: {improvement}")

        # Recalculate the baseline MSE after feature dropping
        final_mse_scores, final_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [target] + all_added_features)
        print(f"Final aggregated MSE after dropping: {final_aggregated_mse}")
        print(f"Final MSE for XGBoost: {final_mse_scores['XGBoost']}")
        print(f"Final MSE for LightGBM: {final_mse_scores['LightGBM']}")

    results.append({
        'target': target,
        'initial_mse_xgboost': baseline_mse_scores['XGBoost'],
        'initial_mse_lightgbm': baseline_mse_scores['LightGBM'],
        'final_mse_xgboost': final_mse_scores['XGBoost'],
        'final_mse_lightgbm': final_mse_scores['LightGBM'],
        'initial_aggregated_mse': aggregated_baseline_mse,
        'final_aggregated_mse': final_aggregated_mse,
        'improvement': aggregated_baseline_mse - final_aggregated_mse,
        'final_feature_space': [target] + all_added_features
    })

    features_df = pd.DataFrame({'features': [target] + all_added_features})
    features_df.to_csv(f'data/engineered/{target}_final_features.csv', index=False)

results_df = pd.DataFrame(results)
results_df.to_csv('feature_engineering_results.csv', index=False)

print("Feature engineering completed for all targets.")



Processing target: FEDFUNDS


""
Date
1976-02-01
1976-03-01
1976-04-01
1976-05-01
1976-06-01
...
2014-01-01
2014-02-01
2014-03-01


,FEDFUNDS
Date,
1976-02-01,0.161211
1976-03-01,-0.097181
1976-04-01,0.030755
1976-05-01,-0.475219
1976-06-01,-0.475219
...,...
2014-01-01,0.003703
2014-02-01,0.000000
2014-03-01,-0.001845


KeyError: "None of [Index(['FEDFUNDS'], dtype='object')] are in the [columns]"

In [1]:
import os
import warnings
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from joblib import Parallel, delayed
# from funcs.engineer_features_funcs import (
#     compute_mse_scores, 
#     evaluate_feature, 
#     compute_mse_with_added_feature, 
#     compute_mse_with_dropped_feature
# )

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from tsfresh import extract_features
from tsfresh.feature_extraction import MinimalFCParameters
from tsfresh.utilities.dataframe_functions import roll_time_series
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from joblib import Parallel, delayed
import optuna
import warnings
from best_params import xgboost_params, lightgbm_params
import logging
    
# Function to optimize parameters using Optuna
def optimize_params(model_name, X_train_scaled, y_train, X_test_scaled, y_test,n_trials):
    def objective(trial):
        if model_name == 'XGBoost':
            params = {
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-5, 1e1),
                'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-5, 1e1)
            }
            model = XGBRegressor(**params)
        elif model_name == 'LightGBM':
            params = {
                'num_leaves': trial.suggest_int('num_leaves', 20, 300),
                'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 20),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
                'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-5, 1e1),
                'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-5, 1e1)
            }
            model = LGBMRegressor(**params)
        else:
            raise ValueError(f"Unknown model name: {model_name}")

        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        return mean_squared_error(y_test, y_pred)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials,n_jobs=1)
    return study.best_params


# Function to compute MSE scores
def compute_mse_scores(X_train, X_test, y_train, y_test, features):
    if not features:  # If no features are provided, return a high MSE (or a default value)
        return {'XGBoost': np.inf, 'LightGBM': np.inf}, np.inf, {}

    X_train = X_train[features].dropna()
    X_test = X_test[features].dropna()
    y_train = y_train.dropna()
    y_test = y_test.dropna()

    X_train_scaled = X_train
    X_test_scaled = X_train

    mse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    return mse_scores, aggregated_mse, {}



# Function to compute baseline MSE scores without adding any new feature
def compute_baseline_mse(X_train, X_test, y_train, y_test, base_features):
    X_train = X_train[base_features].dropna().values
    X_test = X_test[base_features].dropna().values
    y_train = y_train.dropna()
    y_test = y_test.dropna()

    X_train_scaled = X_train
    X_test_scaled = X_train

    mse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    return mse_scores, aggregated_mse

# Function to compute MSE scores after adding a feature
def compute_mse_with_added_feature(X_train, X_test, y_train, y_test, base_features, add_feature):
    X_train = X_train[base_features + [add_feature]].dropna().values
    X_test = X_test[base_features + [add_feature]].dropna().values
    y_train = y_train.dropna().values.ravel()
    y_test = y_test.dropna().values.ravel()

    X_train_scaled = X_train
    X_test_scaled = X_train

    mse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    return mse_scores, aggregated_mse

# Automated Feature Extraction using TSFRESH
def extract_tsfresh_features(data, column_id, column_sort, default_fc_parameters):
    df_long = roll_time_series(data, column_id=column_id, column_sort=column_sort)
    extracted_features = extract_features(df_long, column_id=column_id, column_sort=column_sort, default_fc_parameters=default_fc_parameters)
    extracted_features = extracted_features.dropna(axis=1, how='any')  # Drop columns with NaNs
    return extracted_features

# Function to evaluate a single feature
def evaluate_feature(feature, tsfresh_features_train, tsfresh_features_test, X_train_transformed, X_test_transformed, y_train_transformed, y_test_transformed, base_features, aggregated_baseline_mse, all_added_features):
    if feature in all_added_features:
        return None
    if feature not in tsfresh_features_train.columns or feature not in tsfresh_features_test.columns:
        return None

    temp_X_train = pd.concat([X_train_transformed, tsfresh_features_train[[feature]]], axis=1)
    temp_X_test = pd.concat([X_test_transformed, tsfresh_features_test[[feature]]], axis=1)
    mse_scores, aggregated_mse = compute_mse_with_added_feature(temp_X_train, temp_X_test, y_train_transformed, y_test_transformed, base_features, feature)

    # Handle inf baseline MSE case
    if np.isinf(aggregated_baseline_mse):
        improvement = np.inf if np.isinf(aggregated_mse) else (aggregated_baseline_mse - aggregated_mse)
    else:
        improvement = aggregated_baseline_mse - aggregated_mse

    improvement_status = "improved" if improvement > 0 else "worsened"
    return (feature, aggregated_mse, improvement, improvement_status, mse_scores)

# Function to compute MSE scores after dropping a feature
def compute_mse_with_dropped_feature(X_train, X_test, y_train, y_test, base_features, drop_feature):
    remaining_features = [f for f in base_features if f != drop_feature]
    X_train_dropped = X_train[remaining_features].dropna().values
    X_test_dropped = X_test[remaining_features].dropna().values
    y_train = y_train.dropna().values.ravel()
    y_test = y_test.dropna().values.ravel()

    X_train_scaled = X_train
    X_test_scaled = X_train

    mse_scores = {'XGBoost': [], 'LightGBM': []}
    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    # Handle inf baseline MSE case
    if np.isinf(aggregated_baseline_mse):
        improvement = np.inf if np.isinf(aggregated_mse) else (aggregated_baseline_mse - aggregated_mse)
    else:
        improvement = aggregated_baseline_mse - aggregated_mse

    improvement_status = "improved" if improvement > 0 else "worsened"
    return (drop_feature, aggregated_mse, improvement, mse_scores, improvement_status)


# Function to drop features
# def drop_features(train_combined, target, base_features,aggregated_baseline_mse,threshold):
#     aggregated_mse_scores_dropped = []
#     for feature in base_features:
#         mse_scores, aggregated_mse = compute_mse_with_dropped_feature(X_train_transformed, X_test_transformed, y_train_transformed, y_test_transformed, base_features, feature)
#         improvement = aggregated_baseline_mse - aggregated_mse
#         improvement_status = "improved" if improvement > threshold else "worsened"
#         aggregated_mse_scores_dropped.append((feature, aggregated_mse, improvement, improvement_status, mse_scores))


#     # Sort and drop the least impactful features if they result in improvement
#     aggregated_mse_scores_dropped.sort(key=lambda x: x[1])
#     features_to_drop = [f for f in aggregated_mse_scores_dropped if f[2] > threshold]

#     if not features_to_drop:
#         print("No features were dropped as they did not improve the model.")
#     else:
#         for feature, _, improvement, _, _ in features_to_drop:
#             base_features.remove(feature)
#             print(f"Feature dropped: {feature}, Improvement: {improvement}")

#     print("Feature Dropping Completed.")



# # Suppress warnings
# warnings.filterwarnings("ignore", category=FutureWarning)
# warnings.filterwarnings("ignore", category=UserWarning)

# # Paths for processed data and TSFRESH features
# processed_train_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\processed\train_transformed_combined.csv'
# processed_test_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\processed\test_transformed_combined.csv'
# tsfresh_train_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\train_combined_all_features_filled.csv'
# tsfresh_test_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\test_combined_all_features_filled.csv'

# # Load data
# train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
# test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)
# tsfresh_features_train = pd.read_csv(tsfresh_train_path, index_col='Date', parse_dates=True)
# tsfresh_features_test = pd.read_csv(tsfresh_test_path, index_col='Date', parse_dates=True)

# # Define the base features
# base_features = []

# # Targets to evaluate
# targets = ['FEDFUNDS']

# # Model parameters
# xgboost_params = {'max_depth': 9, 'learning_rate': 0.0168, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5866, 'colsample_bytree': 0.9177, 'reg_alpha': 0.0125, 'reg_lambda': 8.577e-05, 'verbosity': 0}
# lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402, 'colsample_bytree': 0.9405, 'reg_alpha': 0.000756, 'reg_lambda': 0.000259, 'verbosity': -1}

# # Relative threshold for feature dropping
# # Relative threshold for feature dropping
# relative_threshold = 0.002

# # Loop through each target
# results = []

# for target in targets:
#     print(f"\nProcessing target: {target}")
#     X_train = pd.DataFrame(index=train_combined.index)  # Start with empty DataFrame for features
#     X_test = pd.DataFrame(index=test_combined.index)  # Start with empty DataFrame for features
#     y_train = train_combined[[target]]
#     y_test = test_combined[[target]]
    
#     all_added_features = []

#     # Initial baseline MSE calculation (with empty features)
#     baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [])
#     print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
#     print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
#     print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

#     for round_num in range(5):
#         print(f"\n---- Round {round_num + 1} of feature addition ----")

#         # Evaluate adding features
#         results_added = Parallel(n_jobs=-1)(delayed(evaluate_feature)(
#             feature, tsfresh_features_train, tsfresh_features_test, X_train, X_test,
#             y_train, y_test, all_added_features, aggregated_baseline_mse, all_added_features
#         ) for feature in tsfresh_features_train.columns)
        
#         results_added = [res for res in results_added if res is not None]
#         results_added.sort(key=lambda x: x[1])

#         if results_added:
#             print(f"Top features considered for addition: {[f[0] for f in results_added[:3]]}")
        
#         top_to_add = [f for f in results_added[:3] if f[2] > 0]
#         for feature, _, improvement, _, _ in top_to_add:
#             all_added_features.append(feature)
#             X_train[feature] = tsfresh_features_train[feature]
#             X_test[feature] = tsfresh_features_test[feature]
#             print(f"Added feature: {feature} with improvement: {improvement}")

#         # Recalculate the baseline MSE after feature addition
#         baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, all_added_features)
#         print(f"New aggregated MSE after addition: {aggregated_baseline_mse}")
#         print(f"New MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
#         print(f"New MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

#         # Calculate the dynamic threshold for this round
#         threshold = relative_threshold * aggregated_baseline_mse

#         print(f"\n---- Round {round_num + 1} of feature dropping ----")

#         # Evaluate dropping features
#         results_dropped = Parallel(n_jobs=-1)(delayed(compute_mse_with_dropped_feature)(
#             X_train, X_test, y_train, y_test, all_added_features, feature
#         ) for feature in all_added_features)

#         results_dropped.sort(key=lambda x: x[1])

#         if results_dropped:
#             print(f"Features considered for dropping (with their improvements): {[f[0] for f in results_dropped]}")

#         top_to_drop = [f for f in results_dropped if f[2] > threshold]

#         for feature, _, improvement, _, _ in top_to_drop:
#             if improvement > threshold:
#                 all_added_features.remove(feature)
#                 X_train.drop(columns=[feature], inplace=True)
#                 X_test.drop(columns=[feature], inplace=True)
#                 print(f"Dropped feature: {feature} with improvement: {improvement}")

#         # Recalculate the baseline MSE after feature dropping
#         final_mse_scores, final_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, all_added_features)
#         print(f"Final aggregated MSE after dropping: {final_aggregated_mse}")
#         print(f"Final MSE for XGBoost: {final_mse_scores['XGBoost']}")
#         print(f"Final MSE for LightGBM: {final_mse_scores['LightGBM']}")

#     results.append({
#         'target': target,
#         'initial_mse_xgboost': baseline_mse_scores['XGBoost'],
#         'initial_mse_lightgbm': baseline_mse_scores['LightGBM'],
#         'final_mse_xgboost': final_mse_scores['XGBoost'],
#         'final_mse_lightgbm': final_mse_scores['LightGBM'],
#         'initial_aggregated_mse': aggregated_baseline_mse,
#         'final_aggregated_mse': final_aggregated_mse,
#         'improvement': aggregated_baseline_mse - final_aggregated_mse,
#         'final_feature_space': all_added_features
#     })

#     features_df = pd.DataFrame({'features': all_added_features})
#     features_df.to_csv(f'data/engineered/{target}_final_features.csv', index=False)

# results_df = pd.DataFrame(results)
# results_df.to_csv('feature_engineering_results.csv', index=False)

# print("Feature engineering completed for all targets.")



c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import os
import warnings
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from joblib import Parallel, delayed
import optuna
from best_params import xgboost_params, lightgbm_params

# Function to optimize parameters using Optuna
def optimize_params(model_name, X_train_scaled, y_train, X_test_scaled, y_test, n_trials):
    def objective(trial):
        if model_name == 'XGBoost':
            params = {
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-5, 1e1),
                'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-5, 1e1)
            }
            model = XGBRegressor(**params)
        elif model_name == 'LightGBM':
            params = {
                'num_leaves': trial.suggest_int('num_leaves', 20, 300),
                'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 20),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
                'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-5, 1e1),
                'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-5, 1e1)
            }
            model = LGBMRegressor(**params)
        else:
            raise ValueError(f"Unknown model name: {model_name}")

        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        return mean_squared_error(y_test, y_pred)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials, n_jobs=1)
    return study.best_params

# Function to compute MSE scores
def compute_mse_scores(X_train, X_test, y_train, y_test, features):
    if not features:  # If no features are provided, return a high MSE (or a default value)
        return {'XGBoost': np.inf, 'LightGBM': np.inf}, np.inf, {}

    X_train = X_train[features].dropna()
    X_test = X_test[features].dropna()
    y_train = y_train.dropna()
    y_test = y_test.dropna()

    X_train_scaled = X_train
    X_test_scaled = X_train

    mse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    return mse_scores, aggregated_mse, {}

# Function to evaluate a single feature addition
def evaluate_feature(feature, tsfresh_features_train, tsfresh_features_test, X_train_transformed, X_test_transformed, y_train_transformed, y_test_transformed, base_features, aggregated_baseline_mse, all_added_features):
    if feature in all_added_features:
        return None
    if feature not in tsfresh_features_train.columns or feature not in tsfresh_features_test.columns:
        return None

    temp_X_train = pd.concat([X_train_transformed, tsfresh_features_train[[feature]]], axis=1)
    temp_X_test = pd.concat([X_test_transformed, tsfresh_features_test[[feature]]], axis=1)
    mse_scores, aggregated_mse = compute_mse_with_added_feature(temp_X_train, temp_X_test, y_train_transformed, y_test_transformed, base_features, feature)

    improvement = aggregated_baseline_mse - aggregated_mse

    improvement_status = "improved" if improvement > 0 else "worsened"
    return (feature, aggregated_mse, improvement, improvement_status, mse_scores)

# Function to compute MSE scores after dropping a feature
def compute_mse_with_dropped_feature(X_train, X_test, y_train, y_test, base_features, drop_feature, aggregated_baseline_mse):
    remaining_features = [f for f in base_features if f != drop_feature]
    X_train_dropped = X_train[remaining_features].dropna().values
    X_test_dropped = X_test[remaining_features].dropna().values
    y_train = y_train.dropna().values.ravel()
    y_test = y_test.dropna().values.ravel()

    X_train_scaled = X_train
    X_test_scaled = X_train

    mse_scores = {'XGBoost': [], 'LightGBM': []}
    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    improvement = aggregated_baseline_mse - aggregated_mse
    improvement_status = "improved" if improvement > 0 else "worsened"
    return (drop_feature, aggregated_mse, improvement, mse_scores, improvement_status)

# Paths for processed data and TSFRESH features
processed_train_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\processed\train_transformed_combined.csv'
processed_test_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\processed\test_transformed_combined.csv'
tsfresh_train_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\train_combined_all_features_filled.csv'
tsfresh_test_path = r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\test_combined_all_features_filled.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)
tsfresh_features_train = pd.read_csv(tsfresh_train_path, index_col='Date', parse_dates=True)
tsfresh_features_test = pd.read_csv(tsfresh_test_path, index_col='Date', parse_dates=True)

# Define the base features
base_features = []

# Targets to evaluate
targets = ['FEDFUNDS']

# Model parameters
xgboost_params = {'max_depth': 9, 'learning_rate': 0.0168, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5866, 'colsample_bytree': 0.9177, 'reg_alpha': 0.0125, 'reg_lambda': 8.577e-05, 'verbosity': 0}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402, 'colsample_bytree': 0.9405, 'reg_alpha': 0.000756, 'reg_lambda': 0.000259, 'verbosity': -1}

# Relative threshold for feature dropping
relative_threshold = 0.002

# Loop through each target
results = []

for target in targets:
    print(f"\nProcessing target: {target}")
    X_train = pd.DataFrame(index=train_combined.index)  # Start with empty DataFrame for features
    X_test = pd.DataFrame(index=test_combined.index)  # Start with empty DataFrame for features
    y_train = train_combined[[target]].values().ravel()
    y_test = test_combined[[target]].values().ravel()
    
    all_added_features = []

    # Initial baseline MSE calculation (with empty features)
    baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [])
    print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
    print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
    print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

    for round_num in range(5):
        print(f"\n---- Round {round_num + 1} of feature addition ----")

        # Evaluate adding features
        results_added = Parallel(n_jobs=-1)(delayed(evaluate_feature)(
            feature, tsfresh_features_train, tsfresh_features_test, X_train, X_test,
            y_train, y_test, all_added_features, aggregated_baseline_mse, all_added_features
        ) for feature in tsfresh_features_train.columns)
        
        results_added = [res for res in results_added if res is not None]
        results_added.sort(key=lambda x: x[1])

        if results_added:
            print(f"Top features considered for addition: {[f[0] for f in results_added[:3]]}")
        
        top_to_add = [f for f in results_added[:3] if f[2] > 0]
        for feature, _, improvement, _, _ in top_to_add:
            all_added_features.append(feature)
            X_train[feature] = tsfresh_features_train[feature]
            X_test[feature] = tsfresh_features_test[feature]
            print(f"Added feature: {feature} with improvement: {improvement}")

        # Recalculate the baseline MSE after feature addition
        baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, all_added_features)
        print(f"New aggregated MSE after addition: {aggregated_baseline_mse}")
        print(f"New MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
        print(f"New MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

        # Calculate the dynamic threshold for this round
        threshold = relative_threshold * aggregated_baseline_mse

        print(f"\n---- Round {round_num + 1} of feature dropping ----")

        # Evaluate dropping features
        results_dropped = Parallel(n_jobs=-1)(delayed(compute_mse_with_dropped_feature)(
            X_train, X_test, y_train, y_test, all_added_features, feature, aggregated_baseline_mse
        ) for feature in all_added_features)

        results_dropped.sort(key=lambda x: x[1])

        if results_dropped:
            print(f"Features considered for dropping (with their improvements): {[f[0] for f in results_dropped]}")

        top_to_drop = [f for f in results_dropped if f[2] > threshold]

        for feature, _, improvement, _, _ in top_to_drop:
            if improvement > threshold:
                all_added_features.remove(feature)
                X_train.drop(columns=[feature], inplace=True)
                X_test.drop(columns=[feature], inplace=True)
                print(f"Dropped feature: {feature} with improvement: {improvement}")

        # Recalculate the baseline MSE after feature dropping
        final_mse_scores, final_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, all_added_features)
        print(f"Final aggregated MSE after dropping: {final_aggregated_mse}")
        print(f"Final MSE for XGBoost: {final_mse_scores['XGBoost']}")
        print(f"Final MSE for LightGBM: {final_mse_scores['LightGBM']}")

    results.append({
        'target': target,
        'initial_mse_xgboost': baseline_mse_scores['XGBoost'],
        'initial_mse_lightgbm': baseline_mse_scores['LightGBM'],
        'final_mse_xgboost': final_mse_scores['XGBoost'],
        'final_mse_lightgbm': final_mse_scores['LightGBM'],
        'initial_aggregated_mse': aggregated_baseline_mse,
        'final_aggregated_mse': final_aggregated_mse,
        'improvement': aggregated_baseline_mse - final_aggregated_mse,
        'final_feature_space': all_added_features
    })

    features_df = pd.DataFrame({'features': all_added_features})
    features_df.to_csv(f'data/engineered/{target}_final_features.csv', index=False)

results_df = pd.DataFrame(results)
results_df.to_csv('feature_engineering_results.csv', index=False)

print("Feature engineering completed for all targets.")


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Processing target: FEDFUNDS


TypeError: 'numpy.ndarray' object is not callable

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from joblib import Parallel, delayed
import optuna
from best_params import xgboost_params, lightgbm_params

# Function to optimize parameters using Optuna
def optimize_params(model_name, X_train_scaled, y_train, X_test_scaled, y_test, n_trials):
    def objective(trial):
        if model_name == 'XGBoost':
            params = {
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-5, 1e1),
                'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-5, 1e1)
            }
            model = XGBRegressor(**params)
        elif model_name == 'LightGBM':
            params = {
                'num_leaves': trial.suggest_int('num_leaves', 20, 300),
                'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 20),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
                'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-5, 1e1),
                'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-5, 1e1)
            }
            model = LGBMRegressor(**params)
        else:
            raise ValueError(f"Unknown model name: {model_name}")

        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        return mean_squared_error(y_test, y_pred)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials, n_jobs=1)
    return study.best_params

# Function to compute MSE scores
def compute_mse_scores(X_train, X_test, y_train, y_test, features):
    if not features:  # If no features are provided, return a high MSE (or a default value)
        return {'XGBoost': np.inf, 'LightGBM': np.inf}, np.inf, {}

    X_train = X_train[features].dropna()
    X_test = X_test[features].dropna()
    y_train = y_train.dropna()
    y_test = y_test.dropna()

    X_train_scaled = X_train
    X_test_scaled = X_train

    mse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    return mse_scores, aggregated_mse, {}

# Function to evaluate a single feature addition
def evaluate_feature(feature, tsfresh_features_train, tsfresh_features_test, X_train_transformed, X_test_transformed, y_train_transformed, y_test_transformed, base_features, aggregated_baseline_mse, all_added_features):
    if feature in all_added_features:
        return None
    if feature not in tsfresh_features_train.columns or feature not in tsfresh_features_test.columns:
        return None

    temp_X_train = pd.concat([X_train_transformed, tsfresh_features_train[[feature]]], axis=1)
    temp_X_test = pd.concat([X_test_transformed, tsfresh_features_test[[feature]]], axis=1)
    mse_scores, aggregated_mse = compute_mse_with_added_feature(temp_X_train, temp_X_test, y_train_transformed, y_test_transformed, base_features, feature)

    improvement = aggregated_baseline_mse - aggregated_mse

    improvement_status = "improved" if improvement > 0 else "worsened"
    return (feature, aggregated_mse, improvement, improvement_status, mse_scores)

# Function to compute MSE scores after adding a feature
def compute_mse_with_added_feature(X_train, X_test, y_train, y_test, base_features, add_feature):
    X_train = X_train[base_features + [add_feature]].dropna().values
    X_test = X_test[base_features + [add_feature]].dropna().values
    y_train = y_train.dropna().values.ravel()
    y_test = y_test.dropna().values.ravel()

    X_train_scaled = X_train
    X_test_scaled = X_train

    mse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    return mse_scores, aggregated_mse

# Function to compute MSE scores after dropping a feature
def compute_mse_with_dropped_feature(X_train, X_test, y_train, y_test, base_features, drop_feature, aggregated_baseline_mse):
    remaining_features = [f for f in base_features if f != drop_feature]
    X_train_dropped = X_train[remaining_features].dropna().values
    X_test_dropped = X_test[remaining_features].dropna().values
    y_train = y_train.dropna().values.ravel()
    y_test = y_test.dropna().values.ravel()

    X_train_scaled = X_train
    X_test_scaled = X_train

    mse_scores = {'XGBoost': [], 'LightGBM': []}
    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    improvement = aggregated_baseline_mse - aggregated_mse
    improvement_status = "improved" if improvement > 0 else "worsened"
    return (drop_feature, aggregated_mse, improvement, mse_scores, improvement_status)

# Paths for processed data and TSFRESH features
processed_train_path = 'data/processed/train_transformed_combined.csv'
processed_test_path = 'data/processed/test_transformed_combined.csv'
tsfresh_train_path = 'data/tsfresh/train_combined_all_features_filled.csv'
tsfresh_test_path = 'data/tsfresh/test_combined_all_features_filled.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)
tsfresh_features_train = pd.read_csv(tsfresh_train_path, index_col='Date', parse_dates=True)
tsfresh_features_test = pd.read_csv(tsfresh_test_path, index_col='Date', parse_dates=True)

# Define the base features
base_features = []

# Targets to evaluate
targets = ['FEDFUNDS', 'GDP', 'CPIAUCSL', 'CUSR0000SAH1', 'CPILFESL', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'HOUST', 'DSPI', 
           'DGS2', 'DGS5', 'DGS10', 'AAA', 'BAA', 'WTISPLC', 'IMPGS', 'GCE', 'FGCE', 'GDPCTPI', 'PCEPI', 'PCEPILFE', 
           'PAYEMS', 'UNRATE', 'INDPRO', 'CUMFNS', 'USREC']

# Model parameters
xgboost_params = {'max_depth': 9, 'learning_rate': 0.0168, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5866, 'colsample_bytree': 0.9177, 'reg_alpha': 0.0125, 'reg_lambda': 8.577e-05, 'verbosity': 0}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402, 'colsample_bytree': 0.9405, 'reg_alpha': 0.000756, 'reg_lambda': 0.000259, 'verbosity': -1}

# Relative threshold for feature dropping
relative_threshold = 0.002

# Loop through each target
results = []

for target in targets:
    print(f"\nProcessing target: {target}")
    X_train = pd.DataFrame(index=train_combined.index)  # Start with empty DataFrame for features
    X_test = pd.DataFrame(index=test_combined.index)  # Start with empty DataFrame for features
    y_train = train_combined[[target]]
    y_test = test_combined[[target]]
    
    all_added_features = []

    # Initial baseline MSE calculation (with empty features)
    baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [])
    print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
    print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
    print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

    for round_num in range(20):
        print(f"\n---- Round {round_num + 1} of feature addition ----")

        # Evaluate adding features
        results_added = Parallel(n_jobs=-1)(delayed(evaluate_feature)(
            feature, tsfresh_features_train, tsfresh_features_test, X_train, X_test,
            y_train, y_test, all_added_features, aggregated_baseline_mse, all_added_features
        ) for feature in tsfresh_features_train.columns)
        
        results_added = [res for res in results_added if res is not None]
        results_added.sort(key=lambda x: x[1])

        if results_added:
            print(f"Top features considered for addition: {[f[0] for f in results_added[:3]]}")
        
        top_to_add = [f for f in results_added[:3] if f[2] > 0]
        for feature, _, improvement, _, _ in top_to_add:
            all_added_features.append(feature)
            X_train[feature] = tsfresh_features_train[feature]
            X_test[feature] = tsfresh_features_test[feature]
            print(f"Added feature: {feature} with improvement: {improvement}")

        # Recalculate the baseline MSE after feature addition
        baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, all_added_features)
        print(f"New aggregated MSE after addition: {aggregated_baseline_mse}")
        print(f"New MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
        print(f"New MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

        # Calculate the dynamic threshold for this round
        threshold = relative_threshold * aggregated_baseline_mse

        print(f"\n---- Round {round_num + 1} of feature dropping ----")

        # Evaluate dropping features
        results_dropped = Parallel(n_jobs=-1)(delayed(compute_mse_with_dropped_feature)(
            X_train, X_test, y_train, y_test, all_added_features, feature, aggregated_baseline_mse
        ) for feature in all_added_features)

        results_dropped.sort(key=lambda x: x[1])

        if results_dropped:
            print(f"Features considered for dropping (with their improvements): {[f[0] for f in results_dropped]}")

        top_to_drop = [f for f in results_dropped if f[2] > threshold]

        for feature, _, improvement, _, _ in top_to_drop:
            if improvement > threshold:
                all_added_features.remove(feature)
                X_train.drop(columns=[feature], inplace=True)
                X_test.drop(columns=[feature], inplace=True)
                print(f"Dropped feature: {feature} with improvement: {improvement}")

        # Recalculate the baseline MSE after feature dropping
        final_mse_scores, final_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, all_added_features)
        print(f"Final aggregated MSE after dropping: {final_aggregated_mse}")
        print(f"Final MSE for XGBoost: {final_mse_scores['XGBoost']}")
        print(f"Final MSE for LightGBM: {final_mse_scores['LightGBM']}")

    results.append({
        'target': target,
        'initial_mse_xgboost': baseline_mse_scores['XGBoost'],
        'initial_mse_lightgbm': baseline_mse_scores['LightGBM'],
        'final_mse_xgboost': final_mse_scores['XGBoost'],
        'final_mse_lightgbm': final_mse_scores['LightGBM'],
        'initial_aggregated_mse': aggregated_baseline_mse,
        'final_aggregated_mse': final_aggregated_mse,
        'improvement': aggregated_baseline_mse - final_aggregated_mse,
        'final_feature_space': all_added_features
    })

    # Save the final feature list for the target
    features_df = pd.DataFrame({'features': all_added_features})
    features_df.to_csv(f'{target}_final_features.csv', index=False)

# Save the overall results
results_df = pd.DataFrame(results)
results_df.to_csv('feature_engineering_results.csv', index=False)

print("Feature engineering completed for all targets.")


In [1]:
import os
import warnings
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from joblib import Parallel, delayed
import optuna
from best_params import xgboost_params, lightgbm_params

# Function to optimize parameters using Optuna
def optimize_params(model_name, X_train_scaled, y_train, X_test_scaled, y_test, n_trials):
    def objective(trial):
        if model_name == 'XGBoost':
            params = {
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-5, 1e1),
                'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-5, 1e1)
            }
            model = XGBRegressor(**params)
        elif model_name == 'LightGBM':
            params = {
                'num_leaves': trial.suggest_int('num_leaves', 20, 300),
                'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 20),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
                'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-5, 1e1),
                'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-5, 1e1)
            }
            model = LGBMRegressor(**params)
        else:
            raise ValueError(f"Unknown model name: {model_name}")

        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        return mean_squared_error(y_test, y_pred)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials, n_jobs=1)
    return study.best_params

# Function to compute MSE scores
def compute_mse_scores(X_train, X_test, y_train, y_test, features):
    if not features:  # If no features are provided, return a high MSE (or a default value)
        return {'XGBoost': np.inf, 'LightGBM': np.inf}, np.inf, {}

    X_train = X_train[features].dropna()
    X_test = X_test[features].dropna()
    y_train = y_train.dropna()
    y_test = y_test.dropna()

    X_train_scaled = X_train
    X_test_scaled = X_train

    mse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    return mse_scores, aggregated_mse, {}

# Function to evaluate a single feature addition
def evaluate_feature(feature, tsfresh_features_train, tsfresh_features_test, X_train_transformed, X_test_transformed, y_train_transformed, y_test_transformed, base_features, aggregated_baseline_mse, all_added_features):
    if feature in all_added_features:
        return None
    if feature not in tsfresh_features_train.columns or feature not in tsfresh_features_test.columns:
        return None

    temp_X_train = pd.concat([X_train_transformed, tsfresh_features_train[[feature]]], axis=1)
    temp_X_test = pd.concat([X_test_transformed, tsfresh_features_test[[feature]]], axis=1)
    mse_scores, aggregated_mse = compute_mse_with_added_feature(temp_X_train, temp_X_test, y_train_transformed, y_test_transformed, base_features, feature)

    improvement = aggregated_baseline_mse - aggregated_mse

    improvement_status = "improved" if improvement > 0 else "worsened"
    return (feature, aggregated_mse, improvement, improvement_status, mse_scores)

# Function to compute MSE scores after adding a feature
def compute_mse_with_added_feature(X_train, X_test, y_train, y_test, base_features, add_feature):
    X_train = X_train[base_features + [add_feature]].dropna().values
    X_test = X_test[base_features + [add_feature]].dropna().values
    y_train = y_train.dropna().values.ravel()
    y_test = y_test.dropna().values.ravel()

    X_train_scaled = X_train
    X_test_scaled = X_train

    mse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    return mse_scores, aggregated_mse

# Function to compute MSE scores after dropping a feature
def compute_mse_with_dropped_feature(X_train, X_test, y_train, y_test, base_features, drop_feature, aggregated_baseline_mse):
    remaining_features = [f for f in base_features if f != drop_feature]
    X_train_dropped = X_train[remaining_features].dropna().values
    X_test_dropped = X_test[remaining_features].dropna().values
    y_train = y_train.dropna().values.ravel()
    y_test = y_test.dropna().values.ravel()

    # ALREADY SCALED DURING PREPROCESSING
    X_train_scaled = X_train
    X_test_scaled = X_train

    mse_scores = {'XGBoost': [], 'LightGBM': []}
    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    improvement = aggregated_baseline_mse - aggregated_mse
    improvement_status = "improved" if improvement > 0 else "worsened"
    return (drop_feature, aggregated_mse, improvement, mse_scores, improvement_status)

# Paths for processed data and TSFRESH features
processed_train_path = 'data/processed/train_transformed_combined.csv'
processed_test_path = 'data/processed/test_transformed_combined.csv'
tsfresh_train_path = 'data/tsfresh/train_combined_all_features_filled.csv'
tsfresh_test_path = 'data/tsfresh/test_combined_all_features_filled.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)
tsfresh_features_train = pd.read_csv(tsfresh_train_path, index_col='Date', parse_dates=True)
tsfresh_features_test = pd.read_csv(tsfresh_test_path, index_col='Date', parse_dates=True)

# Path to results file
feature_engineering_results = pd.read_csv(r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\feature_engineering_results.csv')

# Define the base features
base_features = []

# Targets to evaluate
targets = ['FEDFUNDS', 'GDP', 'CPIAUCSL', 'CUSR0000SAH1', 'CPILFESL', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'HOUST', 'DSPI', 
           'DGS2', 'DGS5', 'DGS10', 'AAA', 'BAA', 'WTISPLC', 'IMPGS', 'GCE', 'FGCE', 'GDPCTPI', 'PCEPI', 'PCEPILFE', 
           'PAYEMS', 'UNRATE', 'INDPRO', 'CUMFNS', 'USREC']

# Model parameters
xgboost_params = {'max_depth': 9, 'learning_rate': 0.0168, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5866, 'colsample_bytree': 0.9177, 'reg_alpha': 0.0125, 'reg_lambda': 8.577e-05, 'verbosity': 0}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402, 'colsample_bytree': 0.9405, 'reg_alpha': 0.000756, 'reg_lambda': 0.000259, 'verbosity': -1}

# Relative threshold for feature dropping
relative_threshold = 0.002

# Loop through each target
results = []

for target in targets:
    print(f"\nProcessing target: {target}")
    base_features = []
    X_train = pd.DataFrame(index=train_combined.index)  # Start with empty DataFrame for features
    X_test = pd.DataFrame(index=test_combined.index)  # Start with empty DataFrame for features
    y_train = train_combined[[target]]
    y_test = test_combined[[target]]
    
    all_added_features = []

    # Initial baseline MSE calculation (with empty features)
    baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, [])
    print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
    print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
    print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

    for round_num in range(20):
        print(f"\n---- Round {round_num + 1} of feature addition ----")

        # Evaluate adding features
        results_added = Parallel(n_jobs=-1)(delayed(evaluate_feature)(
            feature, tsfresh_features_train, tsfresh_features_test, X_train, X_test,
            y_train, y_test, all_added_features, aggregated_baseline_mse, all_added_features
        ) for feature in tsfresh_features_train.columns)
        
        results_added = [res for res in results_added if res is not None]
        results_added.sort(key=lambda x: x[1])

        if results_added:
            print(f"Top features considered for addition: {[f[0] for f in results_added[:3]]}")
        
        top_to_add = [f for f in results_added[:3] if f[2] > 0]
        for feature, _, improvement, _, _ in top_to_add:
            all_added_features.append(feature)
            X_train[feature] = tsfresh_features_train[feature]
            X_test[feature] = tsfresh_features_test[feature]
            print(f"Added feature: {feature} with improvement: {improvement}")

        # Recalculate the baseline MSE after feature addition
        baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, all_added_features)
        print(f"New aggregated MSE after addition: {aggregated_baseline_mse}")
        print(f"New MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
        print(f"New MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

        # Calculate the dynamic threshold for this round
        threshold = relative_threshold * aggregated_baseline_mse

        print(f"\n---- Round {round_num + 1} of feature dropping ----")

        # Evaluate dropping features
        results_dropped = Parallel(n_jobs=-1)(delayed(compute_mse_with_dropped_feature)(
            X_train, X_test, y_train, y_test, all_added_features, feature, aggregated_baseline_mse
        ) for feature in all_added_features)

        results_dropped.sort(key=lambda x: x[1])

        if results_dropped:
            print(f"Features considered for dropping (with their improvements): {[f[0] for f in results_dropped]}")

        top_to_drop = [f for f in results_dropped if f[2] > threshold]

        for feature, _, improvement, _, _ in top_to_drop:
            if improvement > threshold:
                all_added_features.remove(feature)
                X_train.drop(columns=[feature], inplace=True)
                X_test.drop(columns=[feature], inplace=True)
                print(f"Dropped feature: {feature} with improvement: {improvement}")

        # Recalculate the baseline MSE after feature dropping
        final_mse_scores, final_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, all_added_features)
        print(f"Final aggregated MSE after dropping: {final_aggregated_mse}")
        print(f"Final MSE for XGBoost: {final_mse_scores['XGBoost']}")
        print(f"Final MSE for LightGBM: {final_mse_scores['LightGBM']}")

    results.append({
        'target': target,
        'initial_mse_xgboost': baseline_mse_scores['XGBoost'],
        'initial_mse_lightgbm': baseline_mse_scores['LightGBM'],
        'final_mse_xgboost': final_mse_scores['XGBoost'],
        'final_mse_lightgbm': final_mse_scores['LightGBM'],
        'initial_aggregated_mse': aggregated_baseline_mse,
        'final_aggregated_mse': final_aggregated_mse,
        'improvement': aggregated_baseline_mse - final_aggregated_mse,
        'final_feature_space': all_added_features
    })

    # Save the final feature list for the target
    features_df = pd.DataFrame({'features': all_added_features})
    features_df.to_csv(f'{target}_final_features.csv', index=False)

# Save the overall results
results_df = pd.DataFrame(results)
results_df.to_csv('feature_engineering_results_round2.csv', index=False)

print("Feature engineering completed for all targets.")


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Processing target: FEDFUNDS
Initial aggregated baseline MSE: inf
Initial MSE for XGBoost: inf
Initial MSE for LightGBM: inf

---- Round 1 of feature addition ----


KeyboardInterrupt: 

In [2]:
import os
import warnings
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from joblib import Parallel, delayed
import optuna
from best_params import xgboost_params, lightgbm_params
import re
import ast




# Function to optimize parameters using Optuna
def optimize_params(model_name, X_train_scaled, y_train, X_test_scaled, y_test, n_trials):
    def objective(trial):
        if model_name == 'XGBoost':
            params = {
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-5, 1e1),
                'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-5, 1e1)
            }
            model = XGBRegressor(**params)
        elif model_name == 'LightGBM':
            params = {
                'num_leaves': trial.suggest_int('num_leaves', 20, 300),
                'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 20),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
                'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-5, 1e1),
                'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-5, 1e1)
            }
            model = LGBMRegressor(**params)
        else:
            raise ValueError(f"Unknown model name: {model_name}")

        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        return mean_squared_error(y_test, y_pred)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials, n_jobs=1)
    return study.best_params

# Function to compute MSE scores
def compute_mse_scores(X_train, X_test, y_train, y_test, features):
    if not features:  # If no features are provided, return a high MSE (or a default value)
        return {'XGBoost': np.inf, 'LightGBM': np.inf}, np.inf, {}

    X_train = X_train[features].dropna()
    X_test = X_test[features].dropna()
    y_train = y_train.dropna()
    y_test = y_test.dropna()

    X_train_scaled = X_train
    X_test_scaled = X_train

    mse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    return mse_scores, aggregated_mse, {}

# Function to evaluate a single feature addition
def evaluate_feature(feature, tsfresh_features_train, tsfresh_features_test, X_train_transformed, X_test_transformed, y_train_transformed, y_test_transformed, base_features, aggregated_baseline_mse, all_added_features):
    if feature in all_added_features:
        return None
    if feature not in tsfresh_features_train.columns or feature not in tsfresh_features_test.columns:
        return None

    temp_X_train = pd.concat([X_train_transformed, tsfresh_features_train[[feature]]], axis=1)
    temp_X_test = pd.concat([X_test_transformed, tsfresh_features_test[[feature]]], axis=1)
    mse_scores, aggregated_mse = compute_mse_with_added_feature(temp_X_train, temp_X_test, y_train_transformed, y_test_transformed, base_features, feature)

    improvement = aggregated_baseline_mse - aggregated_mse

    improvement_status = "improved" if improvement > 0 else "worsened"
    return (feature, aggregated_mse, improvement, improvement_status, mse_scores)

# Function to compute MSE scores after adding a feature
def compute_mse_with_added_feature(X_train, X_test, y_train, y_test, base_features, add_feature):
    X_train = X_train[base_features + [add_feature]].dropna().values
    X_test = X_test[base_features + [add_feature]].dropna().values
    y_train = y_train.dropna().values.ravel()
    y_test = y_test.dropna().values.ravel()

    # ALREADY SCALED DURING PREPROCESSING
    X_train_scaled = X_train
    X_test_scaled = X_train

    mse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    return mse_scores, aggregated_mse

# Function to compute MSE scores after dropping a feature
def compute_mse_with_dropped_feature(X_train, X_test, y_train, y_test, base_features, drop_feature, aggregated_baseline_mse):
    remaining_features = [f for f in base_features if f != drop_feature]
    X_train_dropped = X_train[remaining_features].dropna().values
    X_test_dropped = X_test[remaining_features].dropna().values
    y_train = y_train.dropna().values.ravel()
    y_test = y_test.dropna().values.ravel()


    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_dropped)
    X_test_scaled = scaler.transform(X_test_dropped)

    mse_scores = {'XGBoost': [], 'LightGBM': []}
    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    improvement = aggregated_baseline_mse - aggregated_mse
    improvement_status = "improved" if improvement > 0 else "worsened"
    return (drop_feature, aggregated_mse, improvement, mse_scores, improvement_status)

# Paths for processed data and TSFRESH features
processed_train_path = 'data/processed/train_transformed_combined.csv'
processed_test_path = 'data/processed/test_transformed_combined.csv'
tsfresh_train_path = 'data/tsfresh/train_combined_all_features_filled.csv'
tsfresh_test_path = 'data/tsfresh/test_combined_all_features_filled.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)
tsfresh_features_train = pd.read_csv(tsfresh_train_path, index_col='Date', parse_dates=True)
tsfresh_features_test = pd.read_csv(tsfresh_test_path, index_col='Date', parse_dates=True)

# Path to results file
feature_engineering_results = pd.read_csv(r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\feature_engineering_results.csv')

# Define the base features
base_features = []

# Targets to evaluate
targets = ['FEDFUNDS', 'GDP', 'CPIAUCSL', 'CUSR0000SAH1', 'CPILFESL', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'HOUST', 'DSPI', 
           'DGS2', 'DGS5', 'DGS10', 'AAA', 'BAA', 'WTISPLC', 'IMPGS', 'GCE', 'FGCE', 'GDPCTPI', 'PCEPI', 'PCEPILFE', 
           'PAYEMS', 'UNRATE', 'INDPRO', 'CUMFNS', 'USREC']

# Model parameters
xgboost_params = {'max_depth': 9, 'learning_rate': 0.0168, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5866, 'colsample_bytree': 0.9177, 'reg_alpha': 0.0125, 'reg_lambda': 8.577e-05, 'verbosity': 0}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402, 'colsample_bytree': 0.9405, 'reg_alpha': 0.000756, 'reg_lambda': 0.000259, 'verbosity': -1}

# Relative threshold for feature dropping
relative_threshold = 0.002

# Loop through each target
results = []

for target in targets:
    print(f"\nProcessing target: {target}")
    
    # Extract base features for the target
    base_features_str = feature_engineering_results.loc[feature_engineering_results['target'] == target]['final_feature_space'].values[0]
    base_features = ast.literal_eval(base_features_str)

    # Clean the double double-quotes
    base_features = [re.sub(r'""(.*?)""', r'"\1"', feature) for feature in base_features]

    # Verify that these features exist in the tsfresh data
    valid_features = [feature for feature in base_features if feature in tsfresh_features_train.columns]

    X_train = tsfresh_features_train[valid_features]  # Start with valid base features
    X_test = tsfresh_features_test[valid_features]  # Start with valid base features
    y_train = train_combined[[target]]
    y_test = test_combined[[target]]
    
    all_added_features = valid_features.copy()

    # Initial baseline MSE calculation
    baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, valid_features)
    print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
    print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
    print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

    for round_num in range(4):
        print(f"\n---- Round {round_num + 1} of feature addition ----")

        # Evaluate adding features
        results_added = Parallel(n_jobs=-1)(delayed(evaluate_feature)(
            feature, tsfresh_features_train, tsfresh_features_test, X_train, X_test,
            y_train, y_test, all_added_features, aggregated_baseline_mse, all_added_features
        ) for feature in tsfresh_features_train.columns)
        
        results_added = [res for res in results_added if res is not None]
        results_added.sort(key=lambda x: x[1])

        if results_added:
            print(f"Top features considered for addition: {[f[0] for f in results_added[:3]]}")
        
        top_to_add = [f for f in results_added[:3] if f[2] > 0]
        for feature, _, improvement, _, _ in top_to_add:
            all_added_features.append(feature)
            X_train[feature] = tsfresh_features_train[feature]
            X_test[feature] = tsfresh_features_test[feature]
            print(f"Added feature: {feature} with improvement: {improvement}")

        # Recalculate the baseline MSE after feature addition
        baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, all_added_features)
        print(f"New aggregated MSE after addition: {aggregated_baseline_mse}")
        print(f"New MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
        print(f"New MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

        # Calculate the dynamic threshold for this round
        threshold = relative_threshold * aggregated_baseline_mse

        print(f"\n---- Round {round_num + 1} of feature dropping ----")

        # Evaluate dropping features
        results_dropped = Parallel(n_jobs=-1)(delayed(compute_mse_with_dropped_feature)(
            X_train, X_test, y_train, y_test, all_added_features, feature, aggregated_baseline_mse
        ) for feature in all_added_features)

        results_dropped.sort(key=lambda x: x[1])

        if results_dropped:
            print(f"Features considered for dropping (with their improvements): {[f[0] for f in results_dropped]}")

        top_to_drop = [f for f in results_dropped if f[2] > threshold]

        for feature, _, improvement, _, _ in top_to_drop:
            if improvement > threshold:
                all_added_features.remove(feature)
                X_train.drop(columns=[feature], inplace=True)
                X_test.drop(columns=[feature], inplace=True)
                print(f"Dropped feature: {feature} with improvement: {improvement}")

        # Recalculate the baseline MSE after feature dropping
        final_mse_scores, final_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, all_added_features)
        print(f"Final aggregated MSE after dropping: {final_aggregated_mse}")
        print(f"Final MSE for XGBoost: {final_mse_scores['XGBoost']}")
        print(f"Final MSE for LightGBM: {final_mse_scores['LightGBM']}")

    results.append({
        'target': target,
        'initial_mse_xgboost': baseline_mse_scores['XGBoost'],
        'initial_mse_lightgbm': baseline_mse_scores['LightGBM'],
        'final_mse_xgboost': final_mse_scores['XGBoost'],
        'final_mse_lightgbm': final_mse_scores['LightGBM'],
        'initial_aggregated_mse': aggregated_baseline_mse,
        'final_aggregated_mse': final_aggregated_mse,
        'improvement': aggregated_baseline_mse - final_aggregated_mse,
        'final_feature_space': all_added_features
    })

    # Save the final feature list for the target
    features_df = pd.DataFrame({'features': all_added_features})
    features_df.to_csv(f'{target}_final_features.csv', index=False)

# Save the overall results
results_df = pd.DataFrame(results)
results_df.to_csv('feature_engineering_results_round2.csv', index=False)

print("Feature engineering completed for all targets.")
# turn off all warnings
warnings.filterwarnings('ignore')
# FORCE ALLLLL WARNINGS TO STOP IMMEDIEATELY
warnings.filterwarnings(action='ignore')




Processing target: FEDFUNDS


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Initial aggregated baseline MSE: 0.013832248973208266
Initial MSE for XGBoost: 0.007369381570450693
Initial MSE for LightGBM: 0.006462867402757572

---- Round 1 of feature addition ----


KeyboardInterrupt: 

In [47]:
import os
import warnings
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from joblib import Parallel, delayed
import optuna
from best_params import xgboost_params, lightgbm_params
import re
import ast

# Function to compute MSE scores
def compute_mse_scores(X_train, X_test, y_train, y_test, features):
    if not features:  # If no features are provided, return a high MSE (or a default value)
        return {'XGBoost': np.inf, 'LightGBM': np.inf}, np.inf, {}

    X_train = X_train[features].dropna()
    X_test = X_test[features].dropna()
    y_train = y_train.dropna()
    y_test = y_test.dropna()

    print("1. X_train before scaling: ", X_train.shape)
    print("X_test before scaling: ", X_test.shape)
    print("X_train before scaling:NAN ", X_train.isna().sum())
    print("X_test before scaling:NAN ", X_test.isna().sum())
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    print("1. X_train after scaling: ", X_train_scaled.shape)
    print("X_test after scaling: ", X_test_scaled.shape)
    print("X_train after scaling: NAN", np.isnan(X_train_scaled).sum())
    print("X_test after scaling: NAN", np.isnan(X_test_scaled).sum())
    # X_train_scaled = X_train
    # X_test_scaled = X_train

    mse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    return mse_scores, aggregated_mse, {}

# Function to evaluate a single feature addition
def evaluate_feature(feature, tsfresh_features_train, tsfresh_features_test, X_train_transformed, X_test_transformed, y_train_transformed, y_test_transformed, base_features, aggregated_baseline_mse, all_added_features):
    if feature in all_added_features:
        return None
    if feature not in tsfresh_features_train.columns or feature not in tsfresh_features_test.columns:
        return None

    temp_X_train = pd.concat([X_train_transformed, tsfresh_features_train[[feature]]], axis=1)
    temp_X_test = pd.concat([X_test_transformed, tsfresh_features_test[[feature]]], axis=1)
    mse_scores, aggregated_mse = compute_mse_with_added_feature(temp_X_train, temp_X_test, y_train_transformed, y_test_transformed, base_features, feature)

    improvement = aggregated_baseline_mse - aggregated_mse

    improvement_status = "improved" if improvement > 0 else "worsened"
    return (feature, aggregated_mse, improvement, improvement_status, mse_scores)

# Function to compute MSE scores after adding a feature
def compute_mse_with_added_feature(X_train, X_test, y_train, y_test, base_features, add_feature):
    X_train = X_train[base_features + [add_feature]].dropna().values
    X_test = X_test[base_features + [add_feature]].dropna().values
    y_train = y_train.dropna().values.ravel()
    y_test = y_test.dropna().values.ravel()


    X_train_scaled = X_train
    X_test_scaled = X_train


    mse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    return mse_scores, aggregated_mse

# Function to compute MSE scores after dropping a feature
def compute_mse_with_dropped_feature(X_train, X_test, y_train, y_test, base_features, drop_feature, aggregated_baseline_mse):
    remaining_features = [f for f in base_features if f != drop_feature]
    X_train_dropped = X_train[remaining_features].dropna().values
    X_test_dropped = X_test[remaining_features].dropna().values
    y_train = y_train.dropna().values.ravel()
    y_test = y_test.dropna().values.ravel()


    X_train_scaled = X_train_dropped
    X_test_scaled = X_test_dropped

    # X_train_scaled = X_train_dropped
    # X_test_scaled = X_test_dropped

    mse_scores = {'XGBoost': [], 'LightGBM': []}
    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    improvement = aggregated_baseline_mse - aggregated_mse
    improvement_status = "improved" if improvement > 0 else "worsened"
    return (drop_feature, aggregated_mse, improvement, mse_scores, improvement_status)

# Paths for processed data and TSFRESH features
processed_train_path = 'data/processed/train_transformed_combined.csv'
processed_test_path = 'data/processed/test_transformed_combined.csv'
tsfresh_train_path = 'data/tsfresh/train_combined_all_features_filled.csv'
tsfresh_test_path = 'data/tsfresh/test_combined_all_features_filled.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)
tsfresh_features_train = pd.read_csv(tsfresh_train_path, index_col='Date', parse_dates=True)
tsfresh_features_test = pd.read_csv(tsfresh_test_path, index_col='Date', parse_dates=True)
# Path to results file
feature_engineering_results = pd.read_csv(r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\feature_engineering_results.csv')

# Define the base features
base_features = []

# Targets to evaluate
targets = ['FEDFUNDS', 'GDP', 'CPIAUCSL', 'CUSR0000SAH1', 'CPILFESL', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'HOUST', 'DSPI', 
            'DGS5', 'DGS10', 'AAA', 'BAA', 'WTISPLC', 'IMPGS', 'FGCE', 'PCEPI', 'PCEPILFE', 
           'PAYEMS', 'UNRATE', 'INDPRO', 'CUMFNS', 'USREC','DGS2','GCE','GDPCTPI']

# Model parameters
xgboost_params = {'max_depth': 9, 'learning_rate': 0.0168, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5866, 'colsample_bytree': 0.9177, 'reg_alpha': 0.0125, 'reg_lambda': 8.577e-05, 'verbosity': 0}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402, 'colsample_bytree': 0.9405, 'reg_alpha': 0.000756, 'reg_lambda': 0.000259, 'verbosity': -1}

# Relative threshold for feature dropping
relative_threshold = 0.002

# Loop through each target
results = []

for target in targets:
    print(f"\nProcessing target: {target}")
    
    # Extract base features for the target
    base_features_str = feature_engineering_results.loc[feature_engineering_results['target'] == target]['final_feature_space'].values[0]
    base_features = ast.literal_eval(base_features_str)

    # Clean the double double-quotes
    base_features = [re.sub(r'""(.*?)""', r'"\1"', feature) for feature in base_features]

    # Verify that these features exist in the tsfresh data
    valid_features = [feature for feature in base_features if feature in tsfresh_features_train.columns]

    X_train = tsfresh_features_train[valid_features]  # Start with valid base features
    X_test = tsfresh_features_test[valid_features]  # Start with valid base features
    y_train = train_combined[[target]]
    y_test = test_combined[[target]]
    
    all_added_features = valid_features.copy()

    
    # Exclude the target from the list of possible features to add
    possible_features = [feature for feature in train_combined.columns if feature != target]

    train_combined =  train_combined[possible_features]
    test_combined = test_combined[possible_features]

    # Initial baseline MSE calculation
    baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, valid_features)
    variance = np.var(y_test)
    baseline_mse_var = aggregated_baseline_mse / variance

    print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
    print(f"Initial MSE/Variance: {baseline_mse_var}")
    print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
    print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

    for round_num in range(10):
        print(f"\n---- Round {round_num + 1} of feature addition ----")

        # Evaluate adding features
        results_added = Parallel(n_jobs=-1)(delayed(evaluate_feature)(
            feature, train_combined, test_combined, X_train, X_test,
            y_train, y_test, all_added_features, aggregated_baseline_mse, all_added_features
        ) for feature in train_combined.columns)
        
        results_added = [res for res in results_added if res is not None]
        results_added.sort(key=lambda x: x[1])

        if results_added:
            print(f"Top features considered for addition: {[f[0] for f in results_added[:3]]}")
        
        top_to_add = [f for f in results_added[:1] if f[2] > 0]
        for feature, _, improvement, _, _ in top_to_add:
            all_added_features.append(feature)
            X_train[feature] = train_combined[feature]
            X_test[feature] = test_combined[feature]
            print(f"Added feature: {feature} with improvement: {improvement}")
            print(f"% Improvement: {(improvement / aggregated_baseline_mse) * 100}")
            # Calculate improvement in MSE / Variance
            improvement_var = improvement / variance
            print(f"% Improvement in MSE/Variance: {(improvement_var / baseline_mse_var) * 100}")


        # Recalculate the baseline MSE after feature addition
        baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, all_added_features)
        variance = np.var(y_test)
        baseline_mse_var = aggregated_baseline_mse / variance
        print(f"New aggregated MSE after addition: {aggregated_baseline_mse}")
        print(f"New aggregated MSE/Variance: {baseline_mse_var}")
        print(f"New MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
        print(f"New MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

        # Calculate the dynamic threshold for this round
        threshold = relative_threshold * aggregated_baseline_mse

        print(f"\n---- Round {round_num + 1} of feature dropping ----")

        # Evaluate dropping features
        results_dropped = Parallel(n_jobs=-1)(delayed(compute_mse_with_dropped_feature)(
            X_train, X_test, y_train, y_test, all_added_features, feature, aggregated_baseline_mse
        ) for feature in all_added_features)

        results_dropped.sort(key=lambda x: x[1])

        if results_dropped:
            print(f"Features considered for dropping (with their improvements): {[f[0] for f in results_dropped]}")

        top_to_drop = [f for f in results_dropped if f[2] > threshold]

        for feature, _, improvement, _, _ in top_to_drop:
            if improvement > threshold:
                all_added_features.remove(feature)
                X_train.drop(columns=[feature], inplace=True)
                X_test.drop(columns=[feature], inplace=True)
                print(f"Dropped feature: {feature} with improvement: {improvement}")

        # Recalculate the baseline MSE after feature dropping
        final_mse_scores, final_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, all_added_features)
        print(f"Final aggregated MSE after dropping: {final_aggregated_mse}")
        print(f"Final MSE for XGBoost: {final_mse_scores['XGBoost']}")
        print(f"Final MSE for LightGBM: {final_mse_scores['LightGBM']}")

    results.append({
        'target': target,
        'initial_mse_xgboost': baseline_mse_scores['XGBoost'],
        'initial_mse_lightgbm': baseline_mse_scores['LightGBM'],
        'final_mse_xgboost': final_mse_scores['XGBoost'],
        'final_mse_lightgbm': final_mse_scores['LightGBM'],
        'initial_aggregated_mse': aggregated_baseline_mse,
        'final_aggregated_mse': final_aggregated_mse,
        'improvement': aggregated_baseline_mse - final_aggregated_mse,
        'final_feature_space': all_added_features
    })


# Save the overall results
results_df = pd.DataFrame(results)
results_df.to_csv('feature_engineering_results_base_features.csv', index=False)

print("Feature engineering completed for all targets.")
# turn off all warnings
warnings.filterwarnings('ignore')
# FORCE ALLLLL WARNINGS TO STOP IMMEDIEATELY
warnings.filterwarnings(action='ignore')




Processing target: FEDFUNDS
1. X_train before scaling:  (460, 1)
X_test before scaling:  (115, 1)
X_train before scaling:NAN  UNRATE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2    0
dtype: int64
X_test before scaling:NAN  UNRATE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2    0
dtype: int64
1. X_train after scaling:  (460, 1)
X_test after scaling:  (115, 1)
X_train after scaling: NAN 0
X_test after scaling: NAN 0
Initial aggregated baseline MSE: 0.013832248973208266
Initial MSE/Variance: FEDFUNDS    1.833798
dtype: float64
Initial MSE for XGBoost: 0.007369381570450693
Initial MSE for LightGBM: 0.006462867402757572

---- Round 1 of feature addition ----
Top features considered for addition: ['USREC', 'GDPCTPI', 'CUSR0000SAH1']
Added feature: USREC with improvement: 0.0005050418940832913
% Improvement: 3.6511914661275164
% Improvement in MSE/Variance: FEDFUNDS    3.651191
dtype: float64
1. X_train before scaling:  (460, 2)
X_test before scaling:  (115, 2)


KeyboardInterrupt: 

In [49]:
import os
import warnings
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from joblib import Parallel, delayed
import optuna
from best_params import xgboost_params, lightgbm_params
import re
import ast

# Function to compute MSE scores
def compute_mse_scores(X_train, X_test, y_train, y_test, features):
    if not features:  # If no features are provided, return a high MSE (or a default value)
        return {'XGBoost': np.inf, 'LightGBM': np.inf}, np.inf, {}

    X_train = X_train[features].dropna()
    X_test = X_test[features].dropna()
    y_train = y_train.dropna()
    y_test = y_test.dropna()

    # Convert DataFrame to NumPy array without modifying data
    X_train_scaled = X_train.values
    X_test_scaled = X_test.values

    mse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    return mse_scores, aggregated_mse, {}

# Function to evaluate a single feature addition
def evaluate_feature(feature, tsfresh_features_train, tsfresh_features_test, X_train_transformed, X_test_transformed, y_train_transformed, y_test_transformed, base_features, aggregated_baseline_mse, all_added_features):
    if feature in all_added_features:
        return None
    if feature not in tsfresh_features_train.columns or feature not in tsfresh_features_test.columns:
        return None

    temp_X_train = pd.concat([X_train_transformed, tsfresh_features_train[[feature]]], axis=1)
    temp_X_test = pd.concat([X_test_transformed, tsfresh_features_test[[feature]]], axis=1)
    mse_scores, aggregated_mse = compute_mse_with_added_feature(temp_X_train, temp_X_test, y_train_transformed, y_test_transformed, base_features, feature)

    improvement = aggregated_baseline_mse - aggregated_mse

    improvement_status = "improved" if improvement > 0 else "worsened"
    return (feature, aggregated_mse, improvement, improvement_status, mse_scores)

# Function to compute MSE scores after adding a feature
def compute_mse_with_added_feature(X_train, X_test, y_train, y_test, base_features, add_feature):
    X_train = X_train[base_features + [add_feature]].dropna().values
    X_test = X_test[base_features + [add_feature]].dropna().values
    y_train = y_train.dropna().values.ravel()
    y_test = y_test.dropna().values.ravel()

    # Convert DataFrame to NumPy array without modifying data
    X_train_scaled = X_train
    X_test_scaled = X_test


    mse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    return mse_scores, aggregated_mse

# Function to compute MSE scores after dropping a feature
def compute_mse_with_dropped_feature(X_train, X_test, y_train, y_test, base_features, drop_feature, aggregated_baseline_mse):
    remaining_features = [f for f in base_features if f != drop_feature]
    X_train_dropped = X_train[remaining_features].dropna().values
    X_test_dropped = X_test[remaining_features].dropna().values
    y_train = y_train.dropna().values.ravel()
    y_test = y_test.dropna().values.ravel()


    X_train_scaled = X_train_dropped
    X_test_scaled = X_test_dropped

    # X_train_scaled = X_train_dropped
    # X_test_scaled = X_test_dropped

    mse_scores = {'XGBoost': [], 'LightGBM': []}
    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    improvement = aggregated_baseline_mse - aggregated_mse
    improvement_status = "improved" if improvement > 0 else "worsened"
    return (drop_feature, aggregated_mse, improvement, mse_scores, improvement_status)


# Paths for processed data and TSFRESH features
processed_train_path = 'data/processed/train_transformed_combined.csv'
processed_test_path = 'data/processed/test_transformed_combined.csv'
tsfresh_train_path = 'data/tsfresh/train_combined_all_features_filled.csv'
tsfresh_test_path = 'data/tsfresh/test_combined_all_features_filled.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)
tsfresh_features_train = pd.read_csv(tsfresh_train_path, index_col='Date', parse_dates=True)
tsfresh_features_test = pd.read_csv(tsfresh_test_path, index_col='Date', parse_dates=True)
# Path to results file
feature_engineering_results = pd.read_csv(r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\feature_engineering_results.csv')

# Define the base features
base_features = []

# Targets to evaluate
targets = ['FEDFUNDS', 'GDP', 'CPIAUCSL', 'CUSR0000SAH1', 'CPILFESL', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'HOUST', 'DSPI', 
           'DGS5', 'DGS10', 'AAA', 'BAA', 'WTISPLC', 'IMPGS', 'FGCE', 'PCEPI', 'PCEPILFE', 
           'PAYEMS', 'UNRATE', 'INDPRO', 'CUMFNS', 'USREC','DGS2','GCE','GDPCTPI']

# Model parameters
xgboost_params = {'max_depth': 9, 'learning_rate': 0.0168, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5866, 'colsample_bytree': 0.9177, 'reg_alpha': 0.0125, 'reg_lambda': 8.577e-05, 'verbosity': 0}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402, 'colsample_bytree': 0.9405, 'reg_alpha': 0.000756, 'reg_lambda': 0.000259, 'verbosity': -1}

# Run the feature engineering process for each target, as you were before.
# Paths for processed data and TSFRESH features
processed_train_path = 'data/processed/train_transformed_combined.csv'
processed_test_path = 'data/processed/test_transformed_combined.csv'
tsfresh_train_path = 'data/tsfresh/train_combined_all_features_filled.csv'
tsfresh_test_path = 'data/tsfresh/test_combined_all_features_filled.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)
tsfresh_features_train = pd.read_csv(tsfresh_train_path, index_col='Date', parse_dates=True)
tsfresh_features_test = pd.read_csv(tsfresh_test_path, index_col='Date', parse_dates=True)
# Path to results file
feature_engineering_results = pd.read_csv(r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\feature_engineering_results.csv')

# Define the base features
base_features = []

# Targets to evaluate
targets = ['FEDFUNDS', 'GDP', 'CPIAUCSL', 'CUSR0000SAH1', 'CPILFESL', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'HOUST', 'DSPI', 
            'DGS5', 'DGS10', 'AAA', 'BAA', 'WTISPLC', 'IMPGS', 'FGCE', 'PCEPI', 'PCEPILFE', 
           'PAYEMS', 'UNRATE', 'INDPRO', 'CUMFNS', 'USREC','DGS2','GCE','GDPCTPI']

# Model parameters
xgboost_params = {'max_depth': 9, 'learning_rate': 0.0168, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5866, 'colsample_bytree': 0.9177, 'reg_alpha': 0.0125, 'reg_lambda': 8.577e-05, 'verbosity': 0}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402, 'colsample_bytree': 0.9405, 'reg_alpha': 0.000756, 'reg_lambda': 0.000259, 'verbosity': -1}

# Relative threshold for feature dropping
relative_threshold = 0.002

# Loop through each target
results = []

for target in targets:
    print(f"\nProcessing target: {target}")
    
    # Extract base features for the target
    base_features_str = feature_engineering_results.loc[feature_engineering_results['target'] == target]['final_feature_space'].values[0]
    base_features = ast.literal_eval(base_features_str)

    # Clean the double double-quotes
    base_features = [re.sub(r'""(.*?)""', r'"\1"', feature) for feature in base_features]

    # Verify that these features exist in the tsfresh data
    valid_features = [feature for feature in base_features if feature in tsfresh_features_train.columns]

    X_train = tsfresh_features_train[valid_features]  # Start with valid base features
    X_test = tsfresh_features_test[valid_features]  # Start with valid base features
    y_train = train_combined[[target]]
    y_test = test_combined[[target]]
    
    all_added_features = valid_features.copy()

    
    # Exclude the target from the list of possible features to add
    possible_features = [feature for feature in train_combined.columns if feature != target]

    train_combined =  train_combined[possible_features]
    test_combined = test_combined[possible_features]

    # Initial baseline MSE calculation
    baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, valid_features)
    variance = np.var(y_test)
    baseline_mse_var = aggregated_baseline_mse / variance

    print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
    print(f"Initial MSE/Variance: {baseline_mse_var}")
    print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
    print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

    for round_num in range(10):
        print(f"\n---- Round {round_num + 1} of feature addition ----")

        # Evaluate adding features
        results_added = Parallel(n_jobs=-1)(delayed(evaluate_feature)(
            feature, train_combined, test_combined, X_train, X_test,
            y_train, y_test, all_added_features, aggregated_baseline_mse, all_added_features
        ) for feature in train_combined.columns)
        
        results_added = [res for res in results_added if res is not None]
        results_added.sort(key=lambda x: x[1])

        if results_added:
            print(f"Top features considered for addition: {[f[0] for f in results_added[:3]]}")
        
        top_to_add = [f for f in results_added[:1] if f[2] > 0]
        for feature, _, improvement, _, _ in top_to_add:
            all_added_features.append(feature)
            X_train[feature] = train_combined[feature]
            X_test[feature] = test_combined[feature]
            print(f"Added feature: {feature} with improvement: {improvement}")
            print(f"% Improvement: {(improvement / aggregated_baseline_mse) * 100}")
            # Calculate improvement in MSE / Variance
            improvement_var = improvement / variance
            print(f"% Improvement in MSE/Variance: {(improvement_var / baseline_mse_var) * 100}")


        # Recalculate the baseline MSE after feature addition
        baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, all_added_features)
        variance = np.var(y_test)
        baseline_mse_var = aggregated_baseline_mse / variance
        print(f"New aggregated MSE after addition: {aggregated_baseline_mse}")
        print(f"New aggregated MSE/Variance: {baseline_mse_var}")
        print(f"New MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
        print(f"New MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

        # Calculate the dynamic threshold for this round
        threshold = relative_threshold * aggregated_baseline_mse

        print(f"\n---- Round {round_num + 1} of feature dropping ----")

        # Evaluate dropping features
        results_dropped = Parallel(n_jobs=-1)(delayed(compute_mse_with_dropped_feature)(
            X_train, X_test, y_train, y_test, all_added_features, feature, aggregated_baseline_mse
        ) for feature in all_added_features)

        results_dropped.sort(key=lambda x: x[1])

        if results_dropped:
            print(f"Features considered for dropping (with their improvements): {[f[0] for f in results_dropped]}")

        top_to_drop = [f for f in results_dropped if f[2] > threshold]

        for feature, _, improvement, _, _ in top_to_drop:
            if improvement > threshold:
                all_added_features.remove(feature)
                X_train.drop(columns=[feature], inplace=True)
                X_test.drop(columns=[feature], inplace=True)
                print(f"Dropped feature: {feature} with improvement: {improvement}")

        # Recalculate the baseline MSE after feature dropping
        final_mse_scores, final_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, all_added_features)
        print(f"Final aggregated MSE after dropping: {final_aggregated_mse}")
        print(f"Final MSE for XGBoost: {final_mse_scores['XGBoost']}")
        print(f"Final MSE for LightGBM: {final_mse_scores['LightGBM']}")

    results.append({
        'target': target,
        'initial_mse_xgboost': baseline_mse_scores['XGBoost'],
        'initial_mse_lightgbm': baseline_mse_scores['LightGBM'],
        'final_mse_xgboost': final_mse_scores['XGBoost'],
        'final_mse_lightgbm': final_mse_scores['LightGBM'],
        'initial_aggregated_mse': aggregated_baseline_mse,
        'final_aggregated_mse': final_aggregated_mse,
        'improvement': aggregated_baseline_mse - final_aggregated_mse,
        'final_feature_space': all_added_features
    })


# Save the overall results
results_df = pd.DataFrame(results)
results_df.to_csv('feature_engineering_results_base_features.csv', index=False)

print("Feature engineering completed for all targets.")
# turn off all warnings
warnings.filterwarnings('ignore')
# FORCE ALLLLL WARNINGS TO STOP IMMEDIEATELY
warnings.filterwarnings(action='ignore')




Processing target: FEDFUNDS
Initial aggregated baseline MSE: 0.013863269600306573
Initial MSE/Variance: FEDFUNDS    1.83791
dtype: float64
Initial MSE for XGBoost: 0.007398541485511029
Initial MSE for LightGBM: 0.006464728114795543

---- Round 1 of feature addition ----
Top features considered for addition: ['USREC', 'CUSR0000SAH1', 'GDPCTPI']
Added feature: USREC with improvement: 0.0005318021933010675
% Improvement: 3.836051729739911
% Improvement in MSE/Variance: FEDFUNDS    3.836052
dtype: float64
New aggregated MSE after addition: 0.013331467407005505
New aggregated MSE/Variance: FEDFUNDS    1.767407
dtype: float64
New MSE for XGBoost: 0.006866739292209961
New MSE for LightGBM: 0.006464728114795543

---- Round 1 of feature dropping ----
Features considered for dropping (with their improvements): ['USREC', 'UNRATE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2']
Final aggregated MSE after dropping: 0.013331467407005505
Final MSE for XGBoost: 0.006866739292209961
Final

KeyboardInterrupt: 